## Music Recommendation
### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: October 12, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
# TASK 1 (1 PT)
# Show the first row from the training set

import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark.ml.recommendation import ALS

# spark.driver.memory: in local mode, the driver is the executor so this is the memory limit for everything; i have ~48gb so i chose 16gb which leaves plenty of room for my os and vscode
spark = (SparkSession.builder
         .appName("audioscrobbler_als")
         .master("local[*]")
         .config("spark.driver.memory", "16g") 
         .getOrCreate())

# define seed as a constant for use throughout the lab
SEED = 314

# passing explicit schema to save an extra read of file; also ALS requires that user and item ids are integers that fit in 32 bits (declaring IntegerType lets me state that requirement from the start)
schema = StructType([
    StructField("user_id",   IntegerType(), True),
    StructField("artist_id", IntegerType(), True),
    StructField("playcount", IntegerType(), True),
])

raw = spark.read.csv("user_artist_data.txt", sep=" ", schema=schema)

raw.printSchema()
raw.show(5)
# should match wc -l (24,296,858)
print("rows:", raw.count())
# permissive parsing turns bad values into nulls silently, so this must be 0
print("rows with nulls:", raw.filter("user_id IS NULL OR artist_id IS NULL OR playcount IS NULL").count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/16 20:06:40 WARN Utils: Your hostname, Beans-Legion, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/16 20:06:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/16 20:06:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- user_id: integer (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- playcount: integer (nullable = true)

+-------+---------+---------+
|user_id|artist_id|playcount|
+-------+---------+---------+
|1000002|        1|       55|
|1000002|  1000006|       33|
|1000002|  1000007|        8|
|1000002|  1000009|      144|
|1000002|  1000010|      314|
+-------+---------+---------+
only showing top 5 rows
rows: 24296858


rows with nulls: 0


In [2]:
sampled = raw.sample(withReplacement=False, fraction=0.2, seed=SEED)

# count within the sample, since that's the data the model will see
users_10plus = (sampled.groupBy("user_id")
                .agg(F.countDistinct("artist_id").alias("n_artists"))
                .filter(F.col("n_artists") >= 10)
                .select("user_id"))

# left_semi filters by user without adding columns
filtered = sampled.join(users_10plus, on="user_id", how="left_semi")

In [3]:
# cache pins the sample so each split sees the same rows; not guaranteed if partitions get evicted
filtered = filtered.cache()
# cache is lazy, so force it before splitting or the first action recomputes the sample anyway
print("filtered rows:", filtered.count())

train, val, test = filtered.randomSplit([0.6, 0.2, 0.2], seed=SEED)

# sanity check that the weights landed near 60/20/20
print("train:", train.count(), "val:", val.count(), "test:", test.count())

# TASK 1
train.show(1)

filtered rows: 4684468
train: 2810567 val: 936569 test: 937332
+-------+---------+---------+
|user_id|artist_id|playcount|
+-------+---------+---------+
|   7340|      930|        1|
+-------+---------+---------+
only showing top 1 row


In [4]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.

# TASK 2
test.show(1)

# store the id once; a later first() on recomputed data isn't guaranteed to return the same row
first_test = test.first()
test_user_1 = first_test["user_id"]
print("test user 1:", test_user_1)

# als has no factors for users it never trained on, so confirm this one isn't cold start
print("train rows for test user 1:", train.filter(F.col("user_id") == test_user_1).count())
print("test rows for test user 1:", test.filter(F.col("user_id") == test_user_1).count())

+-------+---------+---------+
|user_id|artist_id|playcount|
+-------+---------+---------+
|   7340|     1835|        2|
+-------+---------+---------+
only showing top 1 row
test user 1: 7340
train rows for test user 1: 18
test rows for test user 1: 4


In [5]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# implicitPrefs defaults to false, which would treat play counts as star ratings
# drop keeps nan predictions for unseen users from poisoning any metric
als_1 = ALS(userCol="user_id", itemCol="artist_id", ratingCol="playcount",
            implicitPrefs=True, rank=8, maxIter=5, regParam=0.02, alpha=0.01,
            coldStartStrategy="drop", seed=SEED)

# time one fit so the tuning grid can be sized to what each fit costs
start = time.time()
model_1 = als_1.fit(train)
print(f"model 1 fit: {time.time() - start:.1f}s")

model 1 fit: 5.0s


In [6]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1

def recommend_top_k(model, users_df, k=10):
    """Top-k recommendations per user as (user_id, rank, artist_id, score)."""
    # extra columns or duplicate ids would multiply rows after the explode
    users = users_df.select("user_id").distinct()

    return (model.recommendForUserSubset(users, k)
            # posexplode keeps the model's ordering, which a plain explode would lose
            .select("user_id", F.posexplode("recommendations").alias("pos", "rec"))
            .select("user_id",
                    (F.col("pos") + 1).alias("rank"),
                    F.col("rec.artist_id").alias("artist_id"),
                    F.col("rec.rating").alias("score")))

test_user_1_df = spark.createDataFrame([(test_user_1,)], ["user_id"])

# TASK 3
recs_1 = recommend_top_k(model_1, test_user_1_df, k=10)
recs_1.orderBy("rank").show()

+-------+----+---------+-----------+
|user_id|rank|artist_id|      score|
+-------+----+---------+-----------+
|   7340|   1|     1177| 0.07605168|
|   7340|   2|     1854|0.072421566|
|   7340|   3|      930| 0.06678893|
|   7340|   4|  1000123|0.061134804|
|   7340|   5|     4468|0.058635384|
|   7340|   6|  1000263| 0.05727037|
|   7340|   7|      979|0.056620028|
|   7340|   8|  1000009|0.051327586|
|   7340|   9|       82| 0.04829296|
|   7340|  10|  1270639| 0.04609342|
+-------+----+---------+-----------+



In [ ]:
# TASK 4 (1 PT)
# Briefly explain Recall@K in the context of recommender systems

# see md cell below

Recall@K is equal to the number of relevant items in the top K recommendations divided by the total number of relevant items, analogous to recall (# true positives / # positives). In this lab, (I've interpreted that) an artist is relevant to a user if the user played it at least once in the evaluation set. 

In recommender systems like this one, the model recommends the K items it scores as most relevant, and Recall@K measures what portion of the user’s relevant items it picked up. Because the model can only return K items, the score depends on how many relevant items there are: with 9 relevant items in the top 10, recall is 9/15 when there are 15 relevant items, but only 9/100 when there are 100. (This example is given in the `recommender_systems.ipynb` notebook.)

In [8]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1

def recall_at_k(recs_df, eval_df, k=10):
    """Per-user Recall@K, where relevant = any artist the user played in eval_df."""
    # score only users we recommended for; cold-start users were dropped and have no recs
    rec_users = recs_df.select("user_id").distinct()
    relevant = (eval_df.select("user_id", "artist_id").distinct()
                .join(rec_users, on="user_id", how="left_semi"))

    n_relevant = relevant.groupBy("user_id").agg(F.count("*").alias("n_relevant"))

    hits = (recs_df.filter(F.col("rank") <= k)
            .join(relevant, on=["user_id", "artist_id"], how="inner")
            .groupBy("user_id").agg(F.count("*").alias("n_hits")))

    # left join from the relevant side so users with zero hits still score 0, not vanish
    return (n_relevant.join(hits, on="user_id", how="left")
            .fillna(0, subset=["n_hits"])
            .withColumn("recall_at_k", F.col("n_hits") / F.col("n_relevant")))

# TASK 5
recall_1 = recall_at_k(recs_1, test.filter(F.col("user_id") == test_user_1), k=10)
recall_1.show()

+-------+----------+------+-----------+
|user_id|n_relevant|n_hits|recall_at_k|
+-------+----------+------+-----------+
|   7340|         4|     0|        0.0|
+-------+----------+------+-----------+



In [9]:
# testing pooled and average per user recall

val_users = val.select("user_id").distinct()
print("validation users:", val_users.count())

# time a full evaluation so the grid can be sized to what each point costs
start = time.time()
val_recs_1 = recommend_top_k(model_1, val_users, k=10)
val_recall_1 = recall_at_k(val_recs_1, val, k=10)
val_recall_1.agg(F.mean("recall_at_k").alias("mean_recall"),
                 (F.sum("n_hits") / F.sum("n_relevant")).alias("pooled_recall")).show()
print(f"model 1 validation eval: {time.time() - start:.1f}s")

validation users: 99575


+--------------------+--------------------+
|         mean_recall|       pooled_recall|
+--------------------+--------------------+
|0.020871059175824658|0.015083779198329221|
+--------------------+--------------------+

model 1 validation eval: 120.4s


I'm going with mean.

In [ ]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam) to maximize Recall@K. Fix alpha=0.01. Fit each candidate ALS model on the training data and evaluate its recommendations on the validation set.

# the grid reuses these on every fit and eval, so pay for the split once
train = train.cache()
val = val.cache()
print("train:", train.count(), "val:", val.count())

ranks = [8, 16, 32]
max_iters = [5, 10, 15]
reg_params = [0.01, 0.05, 0.1]

results = []
for rank in ranks:
    for max_iter in max_iters:
        for reg in reg_params:
            start = time.time()
            # alpha stays at 0.01 per the lab instructions; only the three tuned params vary
            als = ALS(userCol="user_id", itemCol="artist_id", ratingCol="playcount",
                      implicitPrefs=True, rank=rank, maxIter=max_iter, regParam=reg,
                      alpha=0.01, coldStartStrategy="drop", seed=SEED)
            model = als.fit(train)
            recs = recommend_top_k(model, val_users, k=10)
            mean_recall = recall_at_k(recs, val, k=10).agg(F.mean("recall_at_k"))=[].first()[0]
            results.append((rank, max_iter, reg, mean_recall))
            print(f"rank={rank:>2} maxIter={max_iter:>2} regParam={reg:<4} "
                  f"mean_recall={mean_recall:.5f} ({time.time() - start:.0f}s)")

train: 2810567 val: 936569


rank= 8 maxIter= 5 regParam=0.01 mean_recall=0.02086 (111s)


rank= 8 maxIter= 5 regParam=0.05 mean_recall=0.02100 (127s)


rank= 8 maxIter= 5 regParam=0.1  mean_recall=0.02175 (133s)


rank= 8 maxIter=10 regParam=0.01 mean_recall=0.02298 (135s)


...

rank=32 maxIter=10 regParam=0.05 mean_recall=0.01994 (164s)


rank=32 maxIter=10 regParam=0.1  mean_recall=0.02098 (164s)


rank=32 maxIter=15 regParam=0.01 mean_recall=0.01987 (158s)


rank=32 maxIter=15 regParam=0.05 mean_recall=0.02022 (159s)


rank=32 maxIter=15 regParam=0.1  mean_recall=0.02170 (168s)


In [11]:
# TASK 6 (2 PTS)
# Report the best values for (rank, maxIter, regParam) along with the Recall@K from the validation set

results_df = spark.createDataFrame(results, ["rank", "maxIter", "regParam", "mean_recall"])
results_df.orderBy(F.desc("mean_recall")).show(27)

# TASK 6
# on a tie, max keeps the first point in grid order, i.e. the smaller rank
best_rank, best_iter, best_reg, best_val_recall = max(results, key=lambda r: r[3])
print(f"best: rank={best_rank}, maxIter={best_iter}, regParam={best_reg}, "
      f"validation mean Recall@10={best_val_recall:.5f}")

+----+-------+--------+--------------------+
|rank|maxIter|regParam|         mean_recall|
+----+-------+--------+--------------------+
|   8|     15|     0.1| 0.02571550591973504|
|   8|     15|    0.05|0.024817322872829974|
|   8|     10|     0.1| 0.02444984230015466|
|  16|     15|     0.1| 0.02428788679262413|
|   8|     15|    0.01| 0.02399523204337105|
|  16|     10|     0.1|0.023914642652302247|
|   8|     10|    0.05| 0.02360876497599377|
|  16|     15|    0.05| 0.02339761986326608|
|  16|     15|    0.01|0.023045689211199365|
|   8|     10|    0.01| 0.02297914364000997|
|  16|     10|    0.05| 0.02295860486569654|
|  16|     10|    0.01|0.022494891113769924|
|  16|      5|     0.1|0.021952163143314007|
|   8|      5|     0.1|0.021746823290039314|
|  32|     15|     0.1|0.021704049747062897|
|   8|      5|    0.05|0.021001703852579726|
|  32|     10|     0.1| 0.02097994561395203|
|   8|      5|    0.01|0.020857086042889085|
|  16|      5|    0.05|0.020667157183898055|
|  32|    

In [12]:
# Retrain the best configuration on train + validation data.

# union matches columns by position, which is safe here because both come from the same split
train_val = train.union(val)

# a fresh fit, since the tuned model never saw validation rows and can't be updated in place
als_2 = ALS(userCol="user_id", itemCol="artist_id", ratingCol="playcount",
            implicitPrefs=True, rank=best_rank, maxIter=best_iter, regParam=best_reg,
            alpha=0.01, coldStartStrategy="drop", seed=SEED)
model_2 = als_2.fit(train_val)

In [13]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.

test_users = test.select("user_id").distinct()
test_recs_2 = recommend_top_k(model_2, test_users, k=10)

# TASK 7
recall_at_k(test_recs_2, test, k=10).agg(F.mean("recall_at_k").alias("test_mean_recall")).show()

+--------------------+
|    test_mean_recall|
+--------------------+
|0.027821874681357123|
+--------------------+



In [14]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?

# TASK 8
recs_2 = recommend_top_k(model_2, test_user_1_df, k=10)
recall_at_k(recs_2, test.filter(F.col("user_id") == test_user_1), k=10).show()

# side by side, so the effect of tuning on this user is visible
(recs_1.select("rank", F.col("artist_id").alias("model_1_artist"))
 .join(recs_2.select("rank", F.col("artist_id").alias("model_2_artist")), on="rank")
 .orderBy("rank")
 .show())

# the user's full listening, labeled by split, to set against what was recommended
activity = (train.withColumn("split", F.lit("train"))
            .unionByName(val.withColumn("split", F.lit("val")))
            .unionByName(test.withColumn("split", F.lit("test")))
            .filter(F.col("user_id") == test_user_1)
            .orderBy(F.desc("playcount")))
activity.show(50)

+-------+----------+------+-----------+
|user_id|n_relevant|n_hits|recall_at_k|
+-------+----------+------+-----------+
|   7340|         4|     0|        0.0|
+-------+----------+------+-----------+

+----+--------------+--------------+
|rank|model_1_artist|model_2_artist|
+----+--------------+--------------+
|   1|          1177|          1854|
|   2|          1854|           930|
|   3|           930|       1001412|
|   4|       1000123|          4267|
|   5|          4468|       1001048|
|   6|       1000263|       1000123|
|   7|           979|       1000031|
|   8|       1000009|       1000263|
|   9|            82|       1007903|
|  10|       1270639|       1006016|
+----+--------------+--------------+

+-------+---------+---------+-----+
|user_id|artist_id|playcount|split|
+-------+---------+---------+-----+
|   7340|  1016821|       24|train|
|   7340|  1000263|       10|train|
|   7340|  1016815|        8|train|
|   7340|     2231|        5|train|
|   7340|  1001332|        3

Test User 1 (7340) has Recall@10 = 0/4 = 0.0 under the tuned model 2, the same as under model 1 in task 5. None of their four test artists (3402, 1835, 1000952, 2105895) appears in either model’s top 10.

The user’s activity helps explain this. User 7340 played 24 artists, most of them only 1–3 times, and the four test artists have only 3, 2, 1, and 1 plays. With such light, spread out listening, there likely was not enough pattern for collaborative filtering to match against other users. Both models also spent two of their ten slots on artists the user already played in training (930 and 1000263). Since they are in the train set, they can’t be test artists, so they can’t count as hits.

Hyperparameter tuning changed the user’s recommendations (6 of the 10 artists differ between the models), and it raised the average validation Recall@10 from 0.0209 (under model 1’s settings) to 0.0257 (rank=8, maxIter=15, regParam=0.1). But it did not help for this user as the recall stayed at 0. This is consistent with Recall@K being averaged across users. Tuning can raise the mean by finding more relevant artists for some users, while a light/diverse listener like 7340 still gets no hits.

In [15]:
# cached data holds memory until released
filtered.unpersist()
train.unpersist()
val.unpersist()

DataFrame[user_id: int, artist_id: int, playcount: int]